In [1]:
import numpy as np
import pandas as pd
import sqlite3
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Load experiment data and get the actual rates
conn = sqlite3.connect('../data/hillstrom.db')
df = pd.read_sql("SELECT * FROM experiment", conn)

treatment = df[df['treatment'] == 'Womens E-Mail']
control = df[df['treatment'] == 'No E-Mail']

n_treatment = len(treatment)
n_control = len(control)
rate_treatment = treatment['visit'].mean()
rate_control = control['visit'].mean()
lift = rate_treatment - rate_control

COST_PER_EMAIL = 0.10  # stated assumption, not a real company figure

In [ ]:
# ==========================================
# 4.1 — Practical Significance
# ==========================================
extra_visits_per_1000 = lift * 1000
cost_per_1000 = 1000 * COST_PER_EMAIL
cost_per_incremental_visit = cost_per_1000 / extra_visits_per_1000

# Mean spend per visitor in the treatment arm
value_per_visit = treatment['spend'].mean() / treatment['visit'].mean()

# Cross-check from Day 3, incremental spend per email sent
incremental_spend_per_email = treatment['spend'].mean() - control['spend'].mean()

print("--- COST ---")
print(f"Extra visits per 1,000 emails:           {extra_visits_per_1000:.1f}")
print(f"Cost to send 1,000 emails:               ${cost_per_1000:.2f}")
print(f"Cost per incremental visit:              ${cost_per_incremental_visit:.2f}")

print("\n--- VALUE ---")
print(f"Value per visit (treatment arm):         ${value_per_visit:.2f}")
print(f"Value / cost ratio:                      {value_per_visit / cost_per_incremental_visit:.1f}x")
print(f"Incremental spend per email sent:        ${incremental_spend_per_email:.4f}")
print(f"Incremental revenue per $1 of email cost: ${incremental_spend_per_email / COST_PER_EMAIL:.2f}")

--- COST ---
Extra visits per 1,000 emails:           45.2
Cost to send 1,000 emails:               $100.00
Cost per incremental visit:              $2.21

--- VALUE ---
Value per visit (treatment arm):         $7.11
Value / cost ratio:                      3.2x
Incremental spend per email sent:        $0.4244
Incremental revenue per $1 of email cost: $4.24


**Practical significance:** Emailing 1,000 customers produces about 45 extra site visits (a 4.52 percentage-point lift) at a cost of $100 under the $0.10-per-email assumption, or roughly $2.21 per incremental visit. An average visitor in the treatment arm spends about $7.11, so an incremental visit is worth roughly 3x its cost. As a cross-check, the Day 3 spend estimate shows about $0.42 of incremental revenue per email sent against $0.10 of cost. Both figures are revenue rather than profit, and incremental visitors may not spend like the average visitor, so this shows the campaign is worth acting on, not what the exact ROI is.

In [ ]:
# ==========================================
# 4.2 — Retrospective Power: Minimum Detectable Effect
# ==========================================
mde_h = NormalIndPower().solve_power(
    effect_size=None,
    nobs1=n_control,
    alpha=0.05,
    power=0.80,
    ratio=1.0,
    alternative='two-sided'
)

# Convert Cohen's h back into a visit rate, then into a lift
mde_rate = np.sin(mde_h / 2 + np.arcsin(np.sqrt(rate_control))) ** 2
mde_lift = mde_rate - rate_control

print("--- MINIMUM DETECTABLE EFFECT (80% power) ---")
print(f"MDE:            {mde_lift * 100:.2f} percentage points")
print(f"Observed lift:  {lift * 100:.2f} percentage points ({lift / mde_lift:.1f}x the MDE)")

# For reference only, power at the observed effect is circular (a function of the p-value)
effect_size = proportion_effectsize(rate_treatment, rate_control)
observed_power = NormalIndPower().power(
    effect_size=effect_size, nobs1=n_control, alpha=0.05, ratio=1.0
)
print(f"\n(Reference) Power at observed effect: {observed_power:.2%}")

--- MINIMUM DETECTABLE EFFECT (80% power) ---
MDE:            0.85 percentage points
Observed lift:  4.52 percentage points (5.3x the MDE)

(Reference) Power at observed effect: 100.00%


**Power:** With about 21,300 customers per arm and a 10.6% baseline visit rate, the test had 80% power to detect a lift as small as 0.85 percentage points. The observed lift of 4.52 percentage points is about five times that, so the experiment's very well powered. Power computed at the observed effect is 100% by construction (it is a function of the p-value), which is why the minimum detectable effect is the more informative figure.

In [4]:
# ==========================================
# 4.3 — Prospective Power
# ==========================================
# Half the observed lift as a more modest, more realistic future effect
half_lift = lift / 2
smaller_effect_size = proportion_effectsize(rate_control + half_lift, rate_control)

req_n = NormalIndPower().solve_power(
    effect_size=smaller_effect_size,
    alpha=0.05,
    power=0.80,
    ratio=1.0,
    alternative='two-sided'
)

print("--- PROSPECTIVE POWER ---")
print(f"Target lift (half observed): {half_lift * 100:.2f} percentage points")
print(f"Required customers per arm:  {int(np.ceil(req_n)):,}")

--- PROSPECTIVE POWER ---
Target lift (half observed): 2.26 percentage points
Required customers per arm:  3,175


**Next test:** Halving the observed lift to 2.26 percentage points, a follow-up test would need roughly 3,175 customers per arm at 80% power. The current sample of about 21,300 per arm is more than sufficient, so a future test could be much smaller and still reliably detect a more modest lift.